In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [1]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_openai import ChatOpenAI
from typing import Callable

# Ollama exposes an OpenAI-compatible API, so configure the local endpoint explicitly.
large_model = ChatOpenAI(
    model="qwen3:4b",
    openai_api_key="ollama",
    openai_api_base="http://localhost:11434/v1"
)
standard_model = ChatOpenAI(
    model="qwen3:4b",
    openai_api_key="ollama",
    openai_api_base="http://localhost:11434/v1"
)


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model=standard_model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [3]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

*(checks watch with a small smile, then glances at the plant in the corner)*  
Oh! Hey—sorry I didn’t water it earlier. I was actually *just* giving it a quick splash this morning when I ran by the breakroom. But yeah, I’m a little behind on the daily plant check-in (it’s my *favorite* thing to do after lunch!).  

**Quick fix:** I’ve got the water bottle ready in my desk drawer right now—just need to grab it and give it a *proper* drink. Should I do it while you’re here? Or... you know what? If you want, I can set up a reminder for tomorrow so it doesn’t happen again. 🌱  

*(taps pen lightly on desk, eyes bright)*  
You’ve got this plant thing down—so if you help me remember *one* day this week, I’ll make sure it’s happy! 😄  

*(pauses)*  
...Need a hand? Or just tell me what time you want it watered? I’m all about it!


In [4]:
print(response["messages"][-1].response_metadata["model_name"])

qwen3:4b


In [5]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

That's actually a great question—most office plants like this (I assume it's a *Spider Plant* or *Pothos*, given the care details) typically need repotting when **roots start showing through the drainage holes** or the pot feels **heavier/less breathable**. 

Since we’ve been rotating it toward the window and it’s been growing steadily (with new leaves!), **we probably don’t need to replace the pot for another 3–4 months**. But here’s what to watch for:  
✅ **Signs it’s time**:  
- Roots are visibly poking out the bottom of the pot  
- The plant feels "spongy" or slow to perk up after watering  
- You notice tiny white "root hairs" around the drainage holes  

If it’s still healthy (like it has), just keep watering gently and rotate it monthly—*that’s* more critical than repotting right now.  

*Pro tip*: When you *do* repot, use a pot **1–2 inches bigger** than current to give roots room to grow. I’d suggest doing it in late spring or early autumn (so it’s not too hot/cold).  

Would 

In [6]:
print(response["messages"][-1].response_metadata["model_name"])

qwen3:4b
